::: {.sb-lesson-links}
[Exercises →](../exercises/house-prices.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/03-pca.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-workflow.ipynb){download="lesson-workflow.ipynb"}
:::

The code of a PCA is short; the choices around it matter more: scaling the data, choosing the number of components, and reading what the components mean. This lesson goes through the whole workflow on a real dataset.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## The data

The wine dataset (bundled with scikit-learn): 178 Italian wines, 13 chemical measurements, and the grape **cultivar** (3 classes). PCA is unsupervised: it will only see the 13 measurements. We keep the cultivar aside to colour the plots and, at the end, to train a model.

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data
cultivar = wine.target.map({0: "cultivar A", 1: "cultivar B", 2: "cultivar C"})
print(X.shape)
X.describe().loc[["mean", "std", "min", "max"]].round(2)

(178, 13)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
mean,13.00,2.34,2.37,19.49,99.74,2.30,2.03,0.36,1.59,5.06,0.96,2.61,746.89
std,0.81,1.12,0.27,3.34,14.28,0.63,1.00,0.12,0.57,2.32,0.23,0.71,314.91
min,11.03,0.74,1.36,10.60,70.00,0.98,0.34,0.13,0.41,1.28,0.48,1.27,278.00
max,14.83,5.80,3.23,30.00,162.00,3.88,5.08,0.66,3.58,13.00,1.71,4.00,1680.00


## PCA is a transformer, like `StandardScaler`

`PCA` works like `StandardScaler`: you `fit` it on the data, then `transform` the data. So it fits in a `Pipeline`.

| Method / attribute | What it does |
|---|---|
| `fit(X)` | computes the mean and the components |
| `transform(X)` | returns the coordinates on the components (one column per component) |
| `fit_transform(X)` | both at once |
| `inverse_transform(Z)` | goes back to the original variables |
| `components_` | the components, one row per component (one column per variable) |
| `explained_variance_` / `explained_variance_ratio_` | the variance of each component / its share of the total |
| `n_components_` | the number of components kept |

## Step 1: standardise first



::: {.callout-tip title="Intuition"}
PCA hunts for variance. A variable measured in large units has a huge variance only because of its unit: `proline` is in the hundreds, `hue` is around 1. Without scaling, PCA would only "hear" the loudest variable. Standardising gives every variable a variance of 1, so they all start with the same voice.
:::



#### See it with data 🔬: PCA without and with scaling

In [ ]:
pca_raw = PCA().fit(X)
X_scaled = StandardScaler().fit_transform(X)
pca = PCA().fit(X_scaled)

print("Variance of each column (raw):", X.var().round(1).sort_values(ascending=False).head(3).to_dict())
for name, model in [("without scaling", pca_raw), ("with scaling   ", pca)]:
    pc1 = pd.Series(model.components_[0], index=X.columns)
    top = pc1.abs().idxmax()
    print(f"{name}: PC1 keeps {model.explained_variance_ratio_[0]:.1%} of the variance, "
          f"its largest weight is {pc1[top]:+.3f} on '{top}'")

Variance of each column (raw): {'proline': 99166.7, 'magnesium': 204.0, 'alcalinity_of_ash': 11.2}
without scaling: PC1 keeps 99.8% of the variance, its largest weight is +1.000 on 'proline'
with scaling   : PC1 keeps 36.2% of the variance, its largest weight is +0.423 on 'flavanoids'


**What this shows:** without scaling, `proline` has a variance of about 99,000, while the next largest is `magnesium` with about 204. PC1 then "keeps 99.8% of the variance", but its weight on `proline` is 1.000 (rounded; the maximum possible for a unit vector): PC1 is just `proline`, and the 12 other variables are ignored. After scaling, PC1 keeps 36.2% of the variance and its largest weight is only 0.423 (on `flavanoids`): it mixes many variables.

**Standardise unless all the variables are in the same unit and their spread matters** (the pixels of an image, for example, all go from 0 to 16).

## Step 2: fit and look at the explained variance

In [ ]:
summary = pd.DataFrame({
    "explained variance (eigenvalue)": pca.explained_variance_,
    "ratio": pca.explained_variance_ratio_,
    "cumulative ratio": pca.explained_variance_ratio_.cumsum(),
}, index=[f"PC{k}" for k in range(1, 14)])
summary.round(3)

,explained variance (eigenvalue),ratio,cumulative ratio
PC1,4.732,0.362,0.362
PC2,2.511,0.192,0.554
PC3,1.454,0.111,0.665
PC4,0.924,0.071,0.736
PC5,0.858,0.066,0.802
PC6,0.645,0.049,0.851
PC7,0.554,0.042,0.893
PC8,0.350,0.027,0.920
PC9,0.291,0.022,0.942
PC10,0.252,0.019,0.962


In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=summary.index, y=summary["ratio"], name="ratio"))
fig.add_trace(go.Scatter(x=summary.index, y=summary["cumulative ratio"], name="cumulative ratio", mode="lines+markers"))
fig.add_hline(y=0.8, line_dash="dot", line_color="#999999", annotation_text="80%")
fig.update_layout(title="Wine: explained variance ratio of each component", yaxis_title="share of the variance",
                  yaxis_range=[0, 1.05], height=400)
fig.show()

**What this shows:** with standardised data, each variable has a variance of 1, so the eigenvalues add up to 13 (scikit-learn's scaler divides by $n$ and PCA by $n-1$, hence 1.006 per variable). PC1 and PC2 keep 36.2% and 19.2% of the variance: 55.4% together. It takes 5 components to pass 80%.

An eigenvalue above 1 means the component carries more variance than a single original variable. Here the first 3 components have an eigenvalue above 1 (the 4th is 0.92).

## Step 3: choose `n_components`

`n_components` accepts three kinds of values:

* an **integer**: the number of components to keep, e.g. `PCA(n_components=2)` to draw the data;
* a **float between 0 and 1**: the share of variance to keep; scikit-learn takes the smallest number of components that reaches it;
* nothing (`None`): all the components.

In [ ]:
for share in [0.8, 0.9, 0.95]:
    p = PCA(n_components=share).fit(X_scaled)
    print(f"PCA(n_components={share}): {p.n_components_} components, "
          f"{p.explained_variance_ratio_.sum():.1%} of the variance")

PCA(n_components=0.8): 5 components, 80.2% of the variance
PCA(n_components=0.9): 8 components, 92.0% of the variance
PCA(n_components=0.95): 10 components, 96.2% of the variance


**What this shows:** asking for 80% gives 5 components (80.2%), 90% gives 8 components (92.0%), and 95% gives 10 components (96.2%). The kept share is always at least the requested one.

## Step 4: read the loadings

Each component is a weighted sum of the standardised variables. The weights, in `components_`, are called the **loadings**. A large weight (positive or negative) means that the variable drives the component.

In [ ]:
loadings = pd.DataFrame(pca.components_[:3].T, index=X.columns, columns=["PC1", "PC2", "PC3"])
fig = px.imshow(loadings.round(2), text_auto=True, color_continuous_scale="RdBu_r", zmin=-0.7, zmax=0.7, aspect="auto",
                title="Loadings of the first 3 components")
fig.update_layout(height=520)
fig.show()

#### See it with data 🔬: a component score, by hand

The coordinate of a wine on PC1 is the sum of its standardised values times the loadings of PC1. Let's check it on the first wine.

In [ ]:
first_wine = X_scaled[0]
by_hand = (first_wine * pca.components_[0]).sum()
print(f"PC1 of the first wine, by hand: {by_hand:.4f}")
print(f"PC1 of the first wine, transform: {pca.transform(X_scaled[:1])[0, 0]:.4f}")

PC1 of the first wine, by hand: 3.3168
PC1 of the first wine, transform: 3.3168


**What this shows:** the weighted sum gives 3.3168, the same value as `transform`. (Here the standardised data already has a mean of 0; in general `transform` subtracts `pca.mean_` first.)

Reading the heatmap:

* **PC1** has large positive weights on `flavanoids` (0.42), `total_phenols` (0.39) and `od280/od315_of_diluted_wines` (0.38), and negative ones on `nonflavanoid_phenols` (−0.30) and `malic_acid` (−0.25). It opposes wines rich in phenols to wines rich in malic acid.
* **PC2** is driven by `color_intensity` (0.53), `alcohol` (0.48) and `proline` (0.36): strong, dark wines have a high PC2.
* **PC3** is mostly `ash` (0.63) and `alcalinity_of_ash` (0.61).

The sign of a component is arbitrary: if all the signs of PC1 were flipped, the interpretation would be the same, reversed.

## Step 5: draw the data on the first 2 components

We draw the wines on PC1 and PC2, coloured by cultivar (which PCA never saw). The arrows show the loadings of 5 variables: a wine lies on the side of the arrows of the variables where it has high values.

In [ ]:
Z = pca.transform(X_scaled)
fig = px.scatter(x=Z[:, 0], y=Z[:, 1], color=cultivar, labels={"x": "PC1 (36.2%)", "y": "PC2 (19.2%)", "color": ""},
                 title="178 wines on their first two principal components")
for var in ["flavanoids", "malic_acid", "color_intensity", "alcohol", "hue"]:
    i = X.columns.get_loc(var)
    tip = 6 * pca.components_[:2, i]                      # loadings, enlarged to be visible
    fig.add_annotation(x=tip[0], y=tip[1], ax=0, ay=0, xref="x", yref="y", axref="x", ayref="y", text="",
                       showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor="#111111")
    fig.add_annotation(x=1.15 * tip[0], y=1.15 * tip[1], text=var, showarrow=False, font=dict(size=12, color="#111111"))
fig.update_layout(height=560, yaxis_scaleanchor="x")
fig.show()

**What this shows:** with only 55.4% of the variance, the three cultivars form three clear groups. Cultivar A sits on the side of `flavanoids` (high PC1) and `alcohol`, cultivar C on the side of `malic_acid` and `color_intensity`, and cultivar B at the bottom, with low PC2 (lighter colour, less alcohol, higher `hue`). A 13-dimensional dataset becomes a picture we can explain with its variables.

## Using PCA before a model

PCA is often a preprocessing step: fewer, uncorrelated variables make many models faster and less prone to overfitting. Two rules:

* **Fit the scaler and the PCA on the training set only**, then apply them to the test set. A `Pipeline` does it for you.
* **The number of components is a hyperparameter**: choose it by cross-validation, like any other.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, wine.target, test_size=0.3, random_state=0, stratify=wine.target)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", LogisticRegression(max_iter=1000)),
])
search = GridSearchCV(pipe, {"pca__n_components": [1, 2, 3, 5, 8, 13]}, cv=5)
search.fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_)[["param_pca__n_components", "mean_test_score"]]
print(results.rename(columns={"param_pca__n_components": "n_components", "mean_test_score": "cross-validated accuracy"})
      .round(3).to_string(index=False))
print("Best:", search.best_params_, f" test accuracy: {search.score(X_test, y_test):.3f}")

 n_components  cross-validated accuracy
            1                     0.831
            2                     0.960
            3                     0.960
            5                     0.968
            8                     0.960
           13                     0.976
Best: {'pca__n_components': 13}  test accuracy: 1.000


**What this shows:** with 1 component, the cross-validated accuracy is 0.831; with 2 components it jumps to 0.960, and it stays between 0.960 and 0.976 with more. The grid search keeps all 13 components (0.976), with a test accuracy of 1.000. On this small, easy dataset PCA does not improve the score, but 2 components already get very close to the 13 variables. On large datasets, such a reduction saves a lot of computing time for a small loss.

## Summary

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),          # 1. standardise (unless all variables share a unit)
    ("pca", PCA(n_components=0.9)),        # 2. an integer, or a share of variance to keep
    ("model", LogisticRegression()),       # 3. optional: any model on the components
])
```

* Without scaling, the variables with the largest units take over the components.
* `explained_variance_ratio_` (and its cumulative sum) tells how much information you keep.
* `components_` holds the loadings: read the largest ones to give a meaning to each component.
* Plot the first 2 components and colour them with a known label to see the structure of the data.
* Before a model, put PCA in a pipeline and choose `n_components` by cross-validation.

::: {.sb-next}
**[Your turn: Principal component analysis exercises →](../exercises/house-prices.ipynb)**
:::